# Regular Expressions (re Module)

A regular expression (regex) is a sequence of characters that describes a search pattern.

Regular expressions are used to search, extract, validate and replace text. Python provides support for them through the built-in `re` module.

Patterns should always be written as raw strings (`r"..."`) so that Python does not treat the backslash `\` as an escape character.

In [5]:
import re

# Match, Search, Findall

| Function | What it does | Returns |
|---|---|---|
| `re.match(p, s)` | Matches only at the **start** of the string | Match object or `None` |
| `re.search(p, s)` | Finds the **first** match anywhere in the string | Match object or `None` |
| `re.findall(p, s)` | Finds **all** matches | List of strings |
| `re.finditer(p, s)` | Finds all matches | Iterator of match objects |
| `re.fullmatch(p, s)` | The **whole** string must match | Match object or `None` |

m.group()     # matched text

m.start()     # starting position

m.end()       # ending position

m.span()      # (start, end)

m.group(1)    # captured group 1

m.groups()   # all captured groups

# re.match()

`re.match()` checks for a match only at the **beginning** of the string. If the pattern is not at the start, it returns `None`.

In [12]:
text = "123 Order shipped, order 456 pending"

print(re.match(r"Order", text))

#the string does not start with a digit
print(re.match(r"\d+", text))

None
<re.Match object; span=(0, 3), match='123'>


# re.search()

`re.search()` scans through the whole string and returns the **first** match found.

In [3]:
text = "Order 123 shipped, order 456 pending"

re.search(r"\d+", text)

<re.Match object; span=(6, 9), match='123'>

# re.findall()

`re.findall()` returns **all** non-overlapping matches as a list of strings.

In [4]:
text = "Order 123 shipped, order 456 pending"

re.findall(r"\d+", text)

['123', '456']

If the pattern contains groups, `findall()` returns the groups instead of the whole match. With more than one group, it returns a list of tuples.

In [16]:
re.findall(r"(\w+)=(\d+)", "aa=1, b = 2, c = 3")

[('aa', '1')]

# re.finditer() and re.fullmatch()

`finditer()` gives match objects, so we also get the position of each match.

`fullmatch()` succeeds only if the **entire** string matches the pattern. It is useful for validation.

In [ ]:
for m in re.finditer(r"\d+", "Order 123 shipped, order 456 pending"):
    print(m.group(), m.span())

123 (6, 9)
456 (25, 28)


In [ ]:
print(re.fullmatch(r"\d{3}-\d{4}", "555-1234"))

print(re.fullmatch(r"\d{3}-\d{4}", "555-1234 extra"))

<re.Match object; span=(0, 8), match='555-1234'>
None


# The Match Object

When `match()` or `search()` succeed they return a match object. Useful methods:

`group()` - the matched text

`group(n)` - the text matched by the n-th group

`groups()` - tuple of all groups

`start()`, `end()`, `span()` - position of the match

In [ ]:
m = re.search(r"(\w+)@(\w+)\.com", "mail: sam@gmail.com")

print(m.group())
print(m.group(1))
print(m.group(2))
print(m.groups())
print(m.start(), m.end(), m.span())

sam@gmail.com
sam
gmail
('sam', 'gmail')
6 19 (6, 19)


If there is no match, the result is `None`. Calling `.group()` on `None` raises an `AttributeError`.

In [ ]:
m = re.search(r"\d+", "no digits here")
print(m.group())

AttributeError: 'NoneType' object has no attribute 'group'

So always check the result before using it.

In [ ]:
m = re.search(r"\d+", "no digits here")

if m:
    print(m.group())
else:
    print("No match found")

#walrus operator makes it shorter
if m := re.search(r"\d+", "abc 42"):
    print(m.group())

#m = re.search(r"\d+", "abc 42")

#if m:

#    print(m.group())

No match found
42


# Sub, Split, Compile

# re.sub()

`re.sub(pattern/condition, replacement, string, count=0)` replaces matches of the pattern with the replacement text.

`count` limits the number of replacements. The default 0 means replace all.

In [ ]:
print(re.sub(r"\d+", "#", "a1b22c333"))

#replace multiple spaces with a single space
print(re.sub(r"\s+", " ", "too    many     spaces"))

#only the first 2 replacements
print(re.sub(r"\d+", "#", "a1b22c333", count=2))

NameError: name 're' is not defined

Groups can be reused in the replacement with backreferences like `\1`, `\2`.

In [ ]:
#swap two words
re.sub(r"(\w+) (\w+)", r"\2 \1", "hello world")

'world hello'

The replacement can also be a **function**. It receives the match object and returns the replacement string.

In [ ]:
#double every number
re.sub(r"\d+", lambda m: str(int(m.group()) * 2), "a5 b10 c20")

'a10 b20 c40'

`re.subn()` works like `sub()` but returns a tuple `(new_string, number_of_replacements)`.

In [ ]:
re.subn(r"\d+", "#", "a1b22c333")

('a#b#c#', 3)

# re.split()

`re.split(pattern, string, maxsplit=0)` splits a string wherever the pattern matches. It is more powerful than `str.split()` because the separator can be a pattern.

In [ ]:
#split on comma or semicolon followed by optional spaces
re.split(r"[,;]\s*", "a, b;c,   d")

['a', 'b;c', '  d']

In [8]:
re.split(r"[,;]\s{3}", "a, b;c,   d")

['a, b;c', 'd']

In [ ]:
#limit the number of splits
re.split(r"\s+", "one two three four", maxsplit=2)

['one', 'two', 'three four']

If the pattern has a capturing group, the separators are also kept in the result.

In [ ]:
re.split(r"(\d)", "a1b2c")

['a', '1', 'b', '2', 'c']

In [ ]:
#split into letters only
re.split(r"\d+", "a1b22c333d")

['a', 'b', 'c', 'd']

In [ ]:
#m = pat.search("call 555-1234 now")

#print(m)
#print(m.group())

#m = pat.search(text)
#if m:
#    print(m.group())

# re.compile()

`re.compile()` converts a pattern into a **pattern object**. It can be reused many times, which makes the code cleaner and faster. The pattern object has the same methods: `match`, `search`, `findall`, `sub`, `split`, etc.

In [ ]:
pat = re.compile(r"\d{3}-\d{4}")

print(pat.search("call 555-1234 now"))
print(pat.findall("555-1234 and 777-9999"))
print(pat.sub("XXX-XXXX", "my number is 555-1234"))

#re.search(r"\d{3}-\d{4}", text)
#re.findall(r"\d{3}-\d{4}", text)
#re.sub(r"\d{3}-\d{4}", replacement, text)

<re.Match object; span=(5, 13), match='555-1234'>
['555-1234', '777-9999']
my number is XXX-XXXX


# Quantifiers and Character Classes

# Quantifiers

Quantifiers specify **how many times** the previous element should repeat.

| Symbol | Meaning |
|---|---|
| `*` | 0 or more |
| `+` | 1 or more |
| `?` | 0 or 1 (optional) |
| `{n}` | exactly n times |
| `{n,}` | n or more times |
| `{n,m}` | between n and m times |

In [ ]:
print(re.findall(r"ab*", "a ab abb abbb"))     # a followed by 0 or more b
print(re.findall(r"ab+", "a ab abb abbb"))     # a followed by 1 or more b
print(re.findall(r"colou?r", "color colour"))  # u is optional

['a', 'ab', 'abb', 'abbb']
['ab', 'abb', 'abbb']
['color', 'colour']


In [ ]:
print(re.findall(r"\d{3}", "12 123 1234"))     # exactly 3 digits
print(re.findall(r"\d{2,}", "1 12 123 1234"))  # 2 or more digits
print(re.findall(r"\d{2,3}", "1 12 123 1234")) # between 2 and 3 digits

#print(re.findall(r"\b\d{2,3}\b", "1 12 123 1234")) #\b is a word boundary, so it will only match 2 or 3 digit numbers that are not part of a larger number

['123', '123']
['12', '123', '1234']
['12', '123', '123']


# Greedy vs Lazy

By default quantifiers are **greedy**: they match as much text as possible.

Adding `?` after a quantifier makes it **lazy**: it matches as little as possible.

In [ ]:
s = "<b>bold</b><i>italic</i>"

#greedy
print(re.findall(r"<.+>", s))

#lazy
print(re.findall(r"<.+?>", s))

['<b>bold</b><i>italic</i>']
['<b>', '</b>', '<i>', '</i>']


# Character Classes

| Pattern | Matches |
|---|---|
| `.` | Any character except a newline |
| `\d` | Any digit (0-9) |
| `\D` | Any non-digit |
| `\w` | Word character (letters, digits, underscore) |
| `\W` | Non-word character |
| `\s` | Whitespace (space, tab, newline) |
| `\S` | Non-whitespace |
| `[abc]` | Any one of a, b or c |
| `[a-z]` | Any character in the range a to z |
| `[^abc]` | Any character **except** a, b, c |

In [ ]:
s = "Hi 99, you_there! #2026"

print(re.findall(r"\d", s))
print(re.findall(r"\D", s))
print(re.findall(r"\w+", s))
print(re.findall(r"\s", s))

['9', '9', '2', '0', '2', '6']
['H', 'i', ' ', ',', ' ', 'y', 'o', 'u', '_', 't', 'h', 'e', 'r', 'e', '!', ' ', '#']
['Hi', '99', 'you_there', '2026']
[' ', ' ', ' ']


In [ ]:
#custom character sets
print(re.findall(r"[aeiou]", "regular expressions"))   # vowels
print(re.findall(r"[a-c1-3]", "a1 b2 c3 d4"))        # ranges can be combined
print(re.findall(r"[^aeiou\s]", "hello world"))      # everything except vowels and spaces

['e', 'u', 'a', 'e', 'e', 'i', 'o']
['a', '1', 'b', '2', 'c', '3']
['h', 'l', 'l', 'w', 'r', 'l', 'd']


Special characters such as `. * + ? ( ) [ ] { } | ^ $ \` must be escaped with a backslash to match them literally.

In [ ]:
print(re.findall(r"a.b", "a.b axb"))      # . matches any character
print(re.findall(r"a\.b", "a.b axb"))     # \. matches only a real dot

['a.b', 'axb']
['a.b']


# Anchors, Groups and Alternation

`^` start of string

`$` end of string

`\b` word boundary

`|` OR

`( )` capturing group

`(?: )` non-capturing group

`(?P<name>...)` named group

In [ ]:
print(re.findall(r"^Hello", "Hello World"))
print(re.findall(r"World$", "Hello World"))
print(re.findall(r"\bcat\b", "cat concat cat."))   # whole word only
print(re.findall(r"cat|dog", "I have a cat and a dog"))

['Hello']
['World']
['cat', 'cat']
['cat', 'dog']


In [ ]:
m = re.search(r"(?P<year>\d{4})-(?P<month>\d{2})-(?P<day>\d{2})", "Date: 2026-10-01")

print(m.group("year"))
print(m.group("month"))
print(m.groupdict())

2026
10
{'year': '2026', 'month': '10', 'day': '01'}


# Flags

Flags change the behaviour of a pattern. They are passed as the last argument. Multiple flags can be combined with `|`.

| Flag | Short | Effect |
|---|---|---|
| `re.IGNORECASE` | `re.I` | Case-insensitive matching |
| `re.MULTILINE` | `re.M` | `^` and `$` match at the start/end of every line |
| `re.DOTALL` | `re.S` | `.` also matches the newline character |
| `re.VERBOSE` | `re.X` | Allows whitespace and comments inside the pattern |
| `re.ASCII` | `re.A` | `\w`, `\d`, `\s` match ASCII characters only |

# re.IGNORECASE

In [ ]:
print(re.findall(r"python", "Python PYTHON python"))

print(re.findall(r"python", "Python PYTHON python", re.I))

['python']
['Python', 'PYTHON', 'python']


# re.MULTILINE

By default `^` and `$` match only at the start and end of the **whole string**. With `re.M` they match at every line.

In [ ]:
text = """Error: disk full
Info: backup done
Error: timeout"""

print(re.findall(r"^Error.*$", text))

print(re.findall(r"^Error.*$", text, re.M))

[]
['Error: disk full', 'Error: timeout']


# re.DOTALL

By default `.` does not match the newline character. With `re.S` it does.

In [ ]:
print(re.search(r"a.b", "a\nb"))

print(re.search(r"a.b", "a\nb", re.S))

None
<re.Match object; span=(0, 3), match='a\nb'>


# re.VERBOSE

`re.X` lets us spread a complex pattern across lines and add comments, which makes it much easier to read.

In [ ]:
phone = re.compile(r"""
    (\d{3})    # area code
    [-.\s]?    # optional separator
    (\d{4})    # number
""", re.X)

phone.findall("555-1234 and 777 9999")

[('555', '1234'), ('777', '9999')]

# Combining Flags

In [ ]:
text = """Hello World
hello python
HELLO regex"""

re.findall(r"^hello", text, re.I | re.M)

['Hello', 'hello', 'HELLO']

Flags can also be written **inline** inside the pattern, e.g. `(?i)` for ignore-case, `(?m)` for multiline, `(?s)` for dotall.

In [ ]:
re.findall(r"(?i)python", "Python PYTHON python")

['Python', 'PYTHON', 'python']

# Python Program to Extract Email Addresses from a String

In [ ]:
text = "Contact us at info@example.com or sales@shop.co.in for details."

emails = re.findall(r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+", text)
print(emails)

['info@example.com', 'sales@shop.co.in']


# Python Program to Validate a Date in DD/MM/YYYY Format

In [ ]:
pattern = re.compile(r"(?P<day>\d{2})/(?P<month>\d{2})/(?P<year>\d{4})")

for d in ["01/10/2026", "1/10/2026", "01-10-2026"]:
    m = pattern.fullmatch(d)
    if m:
        print(d, "-> valid", m.groupdict())
    else:
        print(d, "-> invalid")

01/10/2026 -> valid {'day': '01', 'month': '10', 'year': '2026'}
1/10/2026 -> invalid
01-10-2026 -> invalid


# Python Program to Count Words in a String

In [ ]:
text = "Python is easy, and python is powerful!"

words = re.findall(r"\b\w+\b", text.lower())
print(words)
print(len(words), "words")

['python', 'is', 'easy', 'and', 'python', 'is', 'powerful']
7 words


# Python Program to Remove Special Characters from a String

In [ ]:
text = "He@llo, W#orld! 2026 $"

print(re.sub(r"[^a-zA-Z0-9\s]", "", text))

Hello World 2026 


# Practice Questions

1. Extract all the phone numbers of the form `ddd-dddd` from a string.

2. Replace all the digits in a string with `*`.

3. Split `"a1b22c333d"` so that only the letters remain.

4. Check whether a string is a valid 6 digit PIN code.

5. Find all the lines that start with the word `Error` in a multi-line log.

6. Find all words that start with a capital letter.

In [ ]:
# write your solutions here